# Ejemplos básicos para ir aprendiendo y poner de baseline

A partir de la información del libro `Trading Algorítmico con Python` de Isaac Trullàs

In [1]:
from utils_features import *
import pandas as pd
# import yfinance as yf
import talib as ta
# import re
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import json

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_auc_score, accuracy_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans, DBSCAN

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from arch import arch_model

# import backtrader as bt

In [2]:
spec, df_dia, df_min = load_future('gc1', 'all')

# Análisis técnico

## Indicadores de tendencia

In [3]:
# Media movil simple
def sma(df, n=20):
    df[f"sma_{n}"] = df['close'].rolling(n).mean()
    return df
# Media movil exponencial
def ema(df, n=20):
    df[f"ema_{n}"] = df['close'].ewm(span=n, adjust=False).mean()
    return df

In [4]:
# Convergencia y divergencia de medias moviles
def macd(df, n_fast=12, n_slow=26):
    df['ema_fast'] = df['close'].ewm(span=n_fast, adjust=False).mean()
    df['ema_slow'] = df['close'].ewm(span=n_slow, adjust=False).mean()
    df['macd'] = df['ema_fast'] - df['ema_slow']
    df['signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['histogram'] = df['macd'] - df['signal']
    return df

In [5]:
# Índice direccional medio
def adx(df, n=14):
    df['adx'] = ta.ADX(df['high'], df['low'], df['close'], timeperiod=n)
    return df

## Indicadores de momento

In [6]:
# Índice de fuerza relativa
def rsi(df, n=14):
    df['rsi'] = ta.RSI(df['close'], timeperiod=n)
    return df

In [7]:
# Estocástico
def stoch(df, n=14, slowk_period=3, slowd_period=3, slowk_matype=0, slowd_matype=0):
    df['slowk'], df['slowd'] = ta.STOCH(df['high'], df['low'], df['close'], fastk_period=n, slowk_period=slowk_period, slowd_period=slowd_period, slowk_matype=slowk_matype, slowd_matype=slowd_matype)
    return df

In [8]:
# Índice de fuerza de elder
def elder_force_index(df, n=13):
    df['efi'] = ta.EMA((df['close'] - df['close'].shift(1)) * df['volume'], timeperiod=n)
    return df

## Indicadores de volatilidad

In [9]:
# Bandas de bollinger
def bollinger_bands(df, n=20, num_std_dev=2):
    df['bb_middle'] = df['close'].rolling(n).mean()
    df['bb_std'] = df['close'].rolling(n).std()
    df['bb_upper'] = df['bb_middle'] + num_std_dev * df['bb_std']
    df['bb_lower'] = df['bb_middle'] - num_std_dev * df['bb_std']
    return df

In [10]:
# Rango verdadero medio
def true_range(df, window=14):
    df['atr'] = ta.TRANGE(df['high'], df['low'], df['close'])
    df["range"] = df["high"] - df["low"]
    df["range_mean"] = df["range"].rolling(window).mean()
    return df

In [11]:
# Commodity Channel Index
def cci(df, n=20):
    df['cci'] = ta.CCI(df['high'], df['low'], df['close'], timeperiod=n)
    return df

## Indicadores de volumen

In [12]:
# Volumen en balance
def obv(
    df,
    roc_windows=[5, 10],
    smooth_windows=[10],
    zscore_window=20,
    relative_volume_window=20,
    normalize_by="range",  # "range", "close", "none"
    include_manual=False,
    include_direction=True,
    include_delta=True
):
    # OBV clásico
    df['obv'] = ta.OBV(df['close'], df['volume'])

    # OBV manual opcional
    if include_manual:
        df['obv_manual'] = (
            (df['close'].diff() > 0).astype(int) * df['volume']
            - (df['close'].diff() < 0).astype(int) * df['volume']
        ).cumsum()

    # Normalizaciones
    if normalize_by == "range":
        df['obv_norm'] = df['obv'] / (df['high'] - df['low']).replace(0, np.nan)
    elif normalize_by == "close":
        df['obv_norm'] = df['obv'] / df['close']
    else:
        df['obv_norm'] = df['obv']

    # OBV ROC
    for w in roc_windows:
        df[f'obv_roc_{w}'] = df['obv'].pct_change(w)

    # Suavizados
    for w in smooth_windows:
        df[f'obv_ema_{w}'] = df['obv'].ewm(span=w, adjust=False).mean()
        df[f'obv_sma_{w}'] = df['obv'].rolling(w).mean()

    # Z-score
    mean = df['obv'].rolling(zscore_window).mean()
    std = df['obv'].rolling(zscore_window).std()
    df['obv_z'] = (df['obv'] - mean) / std

    # Dirección del OBV
    if include_direction:
        df['obv_dir'] = df['obv'].diff().apply(lambda x: 1 if x > 0 else -1 if x < 0 else 0)

    # OBV relativo al volumen
    df['obv_rel'] = df['obv'] / df['volume'].rolling(relative_volume_window).sum()

    # Derivada del OBV
    if include_delta:
        df['obv_delta'] = df['obv'].diff()

    return df


In [13]:
# Volumen por precio
def vpt(df):
    df['vpt'] = (df['close'] - df['close'].shift(1)) / df['close'].shift(1) * df['volume']
    return df

In [14]:
# Volumen relativo
def relative_volume(df, window=20):
    df["volumen_medio"] = df["volume"].rolling(window=window).mean()
    df["volumen_relativo"] = df["volume"] / df["volumen_medio"]
    return df

In [15]:
# Acumulación/distribución
def ad(
    df,
    zscore_window=20,
    roc_windows=[5, 10],
    smooth_windows=[10],
    relative_volume_window=20,
    normalize_by="range",   # "range", "close", "none"
    include_clv=True,
    include_delta=True,
    include_direction=True
):
    # --- CLV (Close Location Value) ---
    if include_clv:
        df["clv"] = ((df["close"] - df["low"]) - (df["high"] - df["close"])) / \
                    (df["high"] - df["low"]).replace(0, np.nan)

    # --- A/D clásico ---
    df["ad"] = (df["clv"] * df["volume"]).cumsum()

    # --- Normalización ---
    if normalize_by == "range":
        df["ad_norm"] = df["ad"] / (df["high"] - df["low"]).replace(0, np.nan)
    elif normalize_by == "close":
        df["ad_norm"] = df["ad"] / df["close"]
    else:
        df["ad_norm"] = df["ad"]

    # --- Rate of Change (momentum del A/D) ---
    for w in roc_windows:
        df[f"ad_roc_{w}"] = df["ad"].pct_change(w)

    # --- Suavizados ---
    for w in smooth_windows:
        df[f"ad_ema_{w}"] = df["ad"].ewm(span=w, adjust=False).mean()
        df[f"ad_sma_{w}"] = df["ad"].rolling(w).mean()

    # --- Z-score (acumulación/distribución extrema) ---
    mean = df["ad"].rolling(zscore_window).mean()
    std = df["ad"].rolling(zscore_window).std()
    df["ad_z"] = (df["ad"] - mean) / std

    # --- Derivada del A/D ---
    if include_delta:
        df["ad_delta"] = df["ad"].diff()

    # --- Dirección del A/D ---
    if include_direction:
        df["ad_dir"] = df["ad"].diff().apply(lambda x: 1 if x > 0 else -1 if x < 0 else 0)

    # --- A/D relativo al volumen reciente ---
    df["ad_rel"] = df["ad"] / df["volume"].rolling(relative_volume_window).sum()

    return df

In [16]:
def mfi(df, window=14):
    typical_price = (df['high'] + df['low'] + df['close']) / 3
    money_flow = typical_price * df['volume']

    positive_flow = money_flow.where(typical_price > typical_price.shift(1), 0)
    negative_flow = money_flow.where(typical_price < typical_price.shift(1), 0)

    pos_sum = positive_flow.rolling(window).sum()
    neg_sum = negative_flow.rolling(window).sum()

    df['mfi'] = 100 - (100 / (1 + (pos_sum / neg_sum)))
    return df
def cmf(df, window=20):
    clv = ((df['close'] - df['low']) - (df['high'] - df['close'])) / \
          (df['high'] - df['low']).replace(0, np.nan)

    money_flow_volume = clv * df['volume']

    df['cmf'] = money_flow_volume.rolling(window).sum() / df['volume'].rolling(window).sum()
    return df
def vwap(df):
    typical_price = (df['high'] + df['low'] + df['close']) / 3
    df['vwap'] = (typical_price * df['volume']).cumsum() / df['volume'].cumsum()
    return df
def volume_zscore(df, window=20):
    mean = df['volume'].rolling(window).mean()
    std = df['volume'].rolling(window).std()
    df['volume_z'] = (df['volume'] - mean) / std
    return df
def volume_delta(df):
    df['volume_delta'] = df['volume'].diff()
    return df
def volume_sum(df, window=20):
    df[f'volume_sum_{window}'] = df['volume'].rolling(window).sum()
    return df
def volume_by_range(df):
    df['volume_range'] = df['volume'] / (df['high'] - df['low']).replace(0, np.nan)
    return df
def volume_by_price(df):
    df['volume_price'] = df['volume'] / df['close']
    return df

In [23]:
def historical_volatility(df, window=20):
    returns = np.log(df['close'] / df['close'].shift(1))
    df['hv'] = returns.rolling(window).std() * np.sqrt(252)
    return df
def atr_normalized(df):
    df['atr_norm'] = df['atr'] / df['close']
    return df
def volatility_zscore(df, window=20):
    mean = df['atr'].rolling(window).mean()
    std = df['atr'].rolling(window).std()
    df['atr_z'] = (df['atr'] - mean) / std
    return df
def dema(df, span=20):
    ema = df['close'].ewm(span=span).mean()
    df['dema'] = 2*ema - ema.ewm(span=span).mean()
    return df
def kama(df, window=10, fast=2, slow=30):
    close = df['close'].values

    # Efficiency Ratio (ER)
    change = np.abs(close - np.roll(close, window))
    volatility = np.abs(np.diff(close))
    volatility = np.concatenate([[np.nan], volatility]).astype(float)
    volatility = pd.Series(volatility).rolling(window).sum().values

    er = np.where(volatility == 0, 0, change / volatility)

    # Smoothing Constant (SC)
    fast_sc = 2 / (fast + 1)
    slow_sc = 2 / (slow + 1)
    sc = (er * (fast_sc - slow_sc) + slow_sc) ** 2

    # KAMA calculation (recursive)
    kama = np.zeros_like(close)
    kama[0] = close[0]

    for i in range(1, len(close)):
        kama[i] = kama[i-1] + sc[i] * (close[i] - kama[i-1])

    df['kama'] = kama
    return df
def roc(df, window=10):
    df[f'roc_{window}'] = df['close'].pct_change(window)
    return df
def williams_r(df, window=14):
    highest = df['high'].rolling(window).max()
    lowest = df['low'].rolling(window).min()
    df['williams_r'] = (highest - df['close']) / (highest - lowest)
    return df
def cmo(df, window=14):
    diff = df['close'].diff()
    up = diff.clip(lower=0).rolling(window).sum()
    down = -diff.clip(upper=0).rolling(window).sum()
    df['cmo'] = 100 * (up - down) / (up + down)
    return df
def vol_vol_ratio(df):
    df['vol_vol_ratio'] = df['volume'] / df['atr']
    return df
def vpt_norm(df):
    df['vpt_norm'] = df['vpt'] / df['close']
    return df
def tr_direction(df):
    df['tr_dir'] = np.sign(df['true_range'].diff())
    return df
def candle_body(df):
    df['body'] = (df['close'] - df['open']).abs()
    return df
def wick_ratio(df):
    upper = df['high'] - df[['close','open']].max(axis=1)
    lower = df[['close','open']].min(axis=1) - df['low']
    df['wick_ratio'] = (upper + lower) / (df['high'] - df['low'])
    return df


In [24]:
def generate_features(df, config_json=None):
    """Genera indicadores técnicos en el DataFrame `df`.
    `config_json` puede ser un diccionario o un string JSON con configuraciones por indicador,
    por ejemplo: {'sma': {'n':20}, 'macd': {'n_fast':12,'n_slow':26}}
    Devuelve el DataFrame con nuevas columnas."""
    cfg = {}
    if config_json:
        if isinstance(config_json, str):
            cfg = json.loads(config_json)
        elif isinstance(config_json, dict):
            cfg = config_json
        else:
            raise ValueError("config_json debe ser dict o JSON string")

    # parámetros globales
    global_cfg = cfg.get("global", {})

    # helper para obtener parámetros del indicador
    def p(name):
        local = cfg.get(name, {})
        return {**global_cfg, **local}  # local override

    # función auxiliar
    def _call(func, name, pass_cfg=True):
        params = p(name)
        if pass_cfg and params:
            return func(df, **params)
        return func(df)
    # Aplicar indicadores (se aplican en un orden lógico)
    df = df.copy()
    # Tendencia / medias
    df = _call(sma, 'sma')
    df = _call(ema, 'ema')
    df = _call(dema, 'dema')
    df = _call(kama, 'kama')
    # Momentum / osciladores
    df = _call(macd, 'macd')
    df = _call(rsi, 'rsi')
    df = _call(stoch, 'stoch')
    df = _call(cmo, 'cmo')
    df = _call(williams_r, 'williams_r')
    # Dirección / volatilidad
    df = _call(adx, 'adx')
    df = _call(true_range, 'true_range')
    df = _call(atr_normalized, 'atr_normalized')
    df = _call(volatility_zscore, 'volatility_zscore')
    df = _call(historical_volatility, 'historical_volatility')
    # Volumen y derivados
    df = _call(obv, 'obv')
    df = _call(vpt, 'vpt')
    df = _call(vpt_norm, 'vpt_norm')
    df = _call(relative_volume, 'relative_volume')
    df = _call(ad, 'ad')
    df = _call(mfi, 'mfi')
    df = _call(cmf, 'cmf')
    df = _call(vwap, 'vwap')
    df = _call(volume_zscore, 'volume_zscore')
    df = _call(volume_delta, 'volume_delta')
    df = _call(volume_sum, 'volume_sum')
    df = _call(volume_by_range, 'volume_by_range')
    df = _call(volume_by_price, 'volume_by_price')
    df = _call(vol_vol_ratio, 'vol_vol_ratio')
    # Indicadores técnicos adicionales
    df = _call(cci, 'cci')
    df = _call(bollinger_bands, 'bollinger_bands')
    df = _call(elder_force_index, 'elder_force_index')
    df = _call(candle_body, 'candle_body')
    df = _call(wick_ratio, 'wick_ratio')
    df = _call(roc, 'roc')
    # Normalizaciones / utilidades finales
    df = _call(atr_normalized, 'atr_normalized')
    df = _call(volatility_zscore, 'volatility_zscore')

    return df

In [25]:
df_dia_features = generate_features(df_dia)
df_min_features = generate_features(df_min)

C:\Users\LIJUN\AppData\Local\Temp\ipykernel_11640\4077948443.py:26: RuntimeWarning: invalid value encountered in divide
  er = np.where(volatility == 0, 0, change / volatility)


In [26]:
def resample_ohlcv(df, period="5min"):
    """
    Resamplea un dataframe OHLCV al periodo deseado.
    period puede ser: '1min', '5min', '15min', '30min', '1H', '1D', etc.
    """

    # Asegurar orden temporal
    df = df.sort_values("datetime").copy()

    # Asegurar que datetime es datetime64
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Establecer índice temporal
    df = df.set_index("datetime")

    # Diccionario OHLCV estándar
    ohlc_dict = {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
        "openint": "last"
    }

    # Resample usando el periodo elegido
    df_resampled = df.resample(period).agg(ohlc_dict)

    # Eliminar velas vacías
    df_resampled = df_resampled.dropna(subset=["open", "high", "low", "close"])

    # Añadir columnas extra
    df_resampled["ticker"] = df["ticker"].iloc[0]
    df_resampled["per"] = period

    # Reset index
    df_resampled = df_resampled.reset_index()

    return df_resampled


In [27]:
df_5_min = resample_ohlcv(df_min)

In [28]:
df_5_min_features = generate_features(df_5_min)

In [29]:
# Generar datos ohlcv acumulados diarios a partir de los datos de 5 minutos
def daily_ohlcv_cummulative(df_5_min):
    df = df_5_min.copy()
    
    # Asegurar orden temporal
    df = df.sort_values('datetime')
    
    # Crear columna de día
    df['date'] = df['datetime'].dt.date
    
    # Open del día (primer valor de cada grupo)
    df['open_day'] = df.groupby('date')['open'].transform('first')
    
    # High acumulado intradía
    df['high_cum'] = df.groupby('date')['high'].cummax()
    
    # Low acumulado intradía
    df['low_cum'] = df.groupby('date')['low'].cummin()
    
    # Volume acumulado intradía
    df['volume_cum'] = df.groupby('date')['volume'].cumsum()
    
    # Open interest (último valor hasta ese momento → ya es el actual)
    df['openint_cum'] = df['openint']
    
    # (Opcional) puedes sobrescribir columnas originales
    # df['open'] = df['open_day']
    # df['high'] = df['high_cum']
    # df['low'] = df['low_cum']
    # df['volume'] = df['volume_cum']
    
    # Limpiar columnas auxiliares si quieres
    # df = df.drop(columns=['date', 'open_day', 'high_cum', 'low_cum', 'volume_cum'])
    
    return df

In [30]:
df_dia_cum = daily_ohlcv_cummulative(df_5_min)

In [31]:
df_dia_cum.columns

Index(['datetime', 'open', 'high', 'low', 'close', 'volume', 'openint',
       'ticker', 'per', 'date', 'open_day', 'high_cum', 'low_cum',
       'volume_cum', 'openint_cum'],
      dtype='str')

In [32]:
# 1) copia segura y ordenar
df_acc = df_dia_cum.copy()
df_acc['datetime'] = pd.to_datetime(df_acc['datetime'])
df_acc = df_acc.sort_values('datetime')

# 2) renombrar OHLCV originales a *_5min (para conservarlos)
orig_ohlcv = ['open','high','low','close','volume','openint','ticker','per']
rename_to_5min = {c: f"{c}_5min" for c in orig_ohlcv if c in df_acc.columns}
df_acc = df_acc.rename(columns=rename_to_5min)

# 3) mapear las columnas acumuladas a nombres estándar OHLCV
map_accum = {
    'open_day': 'open',
    'high_cum': 'high',
    'low_cum': 'low',
    'volume_cum': 'volume',
    'openint_cum': 'openint'
}
df_acc = df_acc.rename(columns={k: v for k, v in map_accum.items() if k in df_acc.columns})

# 4) asegurarse de que exista 'close' (usar la close_5min si no hay close_cum)
if 'close' not in df_acc.columns and 'close_5min' in df_acc.columns:
    df_acc['close'] = df_acc['close_5min']

# 5) aplicar generate_features sobre las OHLCV acumuladas
# df_dia_cum_accum_features = generate_features(df_acc)

In [33]:
# df_dia_cum_features = generate_features(df_dia_cum)
df_dia_cum_features = generate_features(df_acc)

In [34]:
# swing_dia = calculate_swings(df_dia, strategy='atr', atr_period=14, atr_mult=1.5)

In [35]:
# df = df_dia_cum_features.join(df_5_min_features.set_index('datetime'), on='datetime', rsuffix='_5min')

In [36]:
df_dia_cum_features.columns

Index(['datetime', 'open_5min', 'high_5min', 'low_5min', 'close_5min',
       'volume_5min', 'openint_5min', 'ticker_5min', 'per_5min', 'date',
       'open', 'high', 'low', 'volume', 'openint', 'close', 'sma_20', 'ema_20',
       'dema', 'kama', 'ema_fast', 'ema_slow', 'macd', 'signal', 'histogram',
       'rsi', 'slowk', 'slowd', 'cmo', 'williams_r', 'adx', 'atr', 'range',
       'range_mean', 'atr_norm', 'atr_z', 'hv', 'obv', 'obv_norm', 'obv_roc_5',
       'obv_roc_10', 'obv_ema_10', 'obv_sma_10', 'obv_z', 'obv_dir', 'obv_rel',
       'obv_delta', 'vpt', 'vpt_norm', 'volumen_medio', 'volumen_relativo',
       'clv', 'ad', 'ad_norm', 'ad_roc_5', 'ad_roc_10', 'ad_ema_10',
       'ad_sma_10', 'ad_z', 'ad_delta', 'ad_dir', 'ad_rel', 'mfi', 'cmf',
       'vwap', 'volume_z', 'volume_delta', 'volume_sum_20', 'volume_range',
       'volume_price', 'vol_vol_ratio', 'cci', 'bb_middle', 'bb_std',
       'bb_upper', 'bb_lower', 'efi', 'body', 'wick_ratio', 'roc_10'],
      dtype='str')

In [37]:
df_5_min_features.columns

Index(['datetime', 'open', 'high', 'low', 'close', 'volume', 'openint',
       'ticker', 'per', 'sma_20', 'ema_20', 'dema', 'kama', 'ema_fast',
       'ema_slow', 'macd', 'signal', 'histogram', 'rsi', 'slowk', 'slowd',
       'cmo', 'williams_r', 'adx', 'atr', 'range', 'range_mean', 'atr_norm',
       'atr_z', 'hv', 'obv', 'obv_norm', 'obv_roc_5', 'obv_roc_10',
       'obv_ema_10', 'obv_sma_10', 'obv_z', 'obv_dir', 'obv_rel', 'obv_delta',
       'vpt', 'vpt_norm', 'volumen_medio', 'volumen_relativo', 'clv', 'ad',
       'ad_norm', 'ad_roc_5', 'ad_roc_10', 'ad_ema_10', 'ad_sma_10', 'ad_z',
       'ad_delta', 'ad_dir', 'ad_rel', 'mfi', 'cmf', 'vwap', 'volume_z',
       'volume_delta', 'volume_sum_20', 'volume_range', 'volume_price',
       'vol_vol_ratio', 'cci', 'bb_middle', 'bb_std', 'bb_upper', 'bb_lower',
       'efi', 'body', 'wick_ratio', 'roc_10'],
      dtype='str')

In [38]:
# Copias locales
a = df_dia_cum_features.copy()
b = df_5_min_features.copy()

# Asegurar datetime y poner índice
a['datetime'] = pd.to_datetime(a['datetime'])
b['datetime'] = pd.to_datetime(b['datetime'])
a = a.set_index('datetime')
b = b.set_index('datetime')

# Índice común para comparar solo filas coincidentes
idx = a.index.intersection(b.index)

cols_a = set(a.columns)
cols_b = set(b.columns)
common = sorted(cols_a & cols_b)
only_a = sorted(cols_a - cols_b)
only_b = sorted(cols_b - cols_a)

identical_exact = []
identical_close = []
different = []

for col in common:
    s1 = a.loc[idx, col]
    s2 = b.loc[idx, col]
    if s1.equals(s2):
        identical_exact.append(col)
        continue
    # intentar comparación numérica con tolerancia (ignora NaNs)
    try:
        x = s1.astype(float)
        y = s2.astype(float)
        mask = ~(x.isna() & y.isna())  # comparar donde no sean NaN ambos
        if mask.any() and np.allclose(x[mask], y[mask], rtol=1e-6, atol=1e-8, equal_nan=True):
            identical_close.append(col)
        else:
            different.append(col)
    except Exception:
        different.append(col)

print('Sólo en df_dia_cum_features:', only_a)
print('Sólo en df_5_min_features:', only_b)
print('Idénticas (exact):', identical_exact)
print('Idénticas (numéricas, close):', identical_close)
print('Diferentes:', different)

Sólo en df_dia_cum_features: ['close_5min', 'date', 'high_5min', 'low_5min', 'open_5min', 'openint_5min', 'per_5min', 'ticker_5min', 'volume_5min']
Sólo en df_5_min_features: ['per', 'ticker']
Idénticas (exact): ['bb_lower', 'bb_middle', 'bb_std', 'bb_upper', 'close', 'cmo', 'dema', 'ema_20', 'ema_fast', 'ema_slow', 'histogram', 'hv', 'kama', 'macd', 'obv_dir', 'openint', 'roc_10', 'rsi', 'signal', 'sma_20']
Idénticas (numéricas, close): []
Diferentes: ['ad', 'ad_delta', 'ad_dir', 'ad_ema_10', 'ad_norm', 'ad_rel', 'ad_roc_10', 'ad_roc_5', 'ad_sma_10', 'ad_z', 'adx', 'atr', 'atr_norm', 'atr_z', 'body', 'cci', 'clv', 'cmf', 'efi', 'high', 'low', 'mfi', 'obv', 'obv_delta', 'obv_ema_10', 'obv_norm', 'obv_rel', 'obv_roc_10', 'obv_roc_5', 'obv_sma_10', 'obv_z', 'open', 'range', 'range_mean', 'slowd', 'slowk', 'vol_vol_ratio', 'volume', 'volume_delta', 'volume_price', 'volume_range', 'volume_sum_20', 'volume_z', 'volumen_medio', 'volumen_relativo', 'vpt', 'vpt_norm', 'vwap', 'wick_ratio', 'wi

In [39]:
# columnas a excluir (ohlcv y metadata)
exclude = ['open','high','low','close','volume','openint','ticker','per','date']

# seleccionar sólo features útiles del 5min
features_5min = [c for c in df_5_min_features.columns if c not in exclude and c != 'datetime']

# join manteniendo las OHLCV acumuladas
df_final = df_dia_cum_features.join(
    df_5_min_features.set_index('datetime')[features_5min],
    on='datetime',
    rsuffix='_5min'
)

# eliminar duplicados exactos (si una columna _5min es idéntica a la original)
for col in list(df_dia_cum_features.columns):
    col5 = f"{col}_5min"
    if col5 in df_final.columns:
        try:
            if df_final[col].equals(df_final[col5]):
                df_final.drop(columns=[col5], inplace=True)
        except Exception:
            pass

# Opcional: revisar resultado
# print("Columns:", df_final.columns.tolist())

In [40]:
df_final.columns

Index(['datetime', 'open_5min', 'high_5min', 'low_5min', 'volume_5min',
       'ticker_5min', 'per_5min', 'date', 'open', 'high',
       ...
       'volume_z_5min', 'volume_delta_5min', 'volume_sum_20_5min',
       'volume_range_5min', 'volume_price_5min', 'vol_vol_ratio_5min',
       'cci_5min', 'efi_5min', 'body_5min', 'wick_ratio_5min'],
      dtype='str', length=124)

In [42]:
df_final.loc[0]

datetime              2010-09-20 07:00:00
open_5min                          1279.8
high_5min                          1280.7
low_5min                           1279.8
volume_5min                            72
                             ...         
vol_vol_ratio_5min                    NaN
cci_5min                              NaN
efi_5min                              NaN
body_5min                             0.0
wick_ratio_5min                       1.0
Name: 0, Length: 124, dtype: object

In [44]:
df_final.loc[100].to_dict()

{'datetime': Timestamp('2010-09-20 15:20:00'),
 'open_5min': 1282.7,
 'high_5min': 1282.7,
 'low_5min': 1281.8,
 'volume_5min': 498,
 'ticker_5min': 'GC',
 'per_5min': '5min',
 'date': datetime.date(2010, 9, 20),
 'open': 1279.8,
 'high': 1284.9,
 'low': 1279.8,
 'volume': 15656,
 'openint': 0.0,
 'close': 1282.3,
 'sma_20': 1282.325,
 'ema_20': 1282.3109799445162,
 'dema': 1282.4056759139078,
 'kama': nan,
 'ema_fast': 1282.3537879392225,
 'ema_slow': 1282.2908021770268,
 'macd': 0.0629857621956944,
 'signal': 0.10814895106774601,
 'histogram': -0.045163188872051604,
 'rsi': 50.10317296913942,
 'slowk': 44.4444444444429,
 'slowd': 42.26579520697021,
 'cmo': -5.882352941178044,
 'williams_r': 0.5098039215686405,
 'adx': 100.0,
 'atr': 5.100000000000136,
 'range': 5.100000000000136,
 'range_mean': 5.100000000000136,
 'atr_norm': 0.003977228417687075,
 'atr_z': nan,
 'hv': 0.007255594252264098,
 'obv': -102855.0,
 'obv_norm': -20167.64705882299,
 'obv_roc_5': 0.7474812687949166,
 'obv_ro

In [ ]:
df = df_final.copy()

# Modelos predictivos

## Regresión lineal y logística

In [ ]:
# Regresion lineal
X = df.drop(columns=["close"])
Y = df["close"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2}")



In [ ]:
# Regresion logistica
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
X = df.drop(columns=["close", "target"])
Y = df["target"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)

preccision = accuracy_score(Y_test, Y_pred)
matriz_confusion = confusion_matrix(Y_test, Y_pred)
roc_auc = roc_auc_score(Y_test, model.predict(X_test)[:, 1])

print(f"Accuracy: {preccision}")
print(f"Confusion Matrix:\n{matriz_confusion}")
print(f"ROC AUC Score: {roc_auc}")

## Modelos de series temporales

In [ ]:
# ARIMA
# df.reset_index()
dates = df["date"]
close = df["close"]
close.index = range(len(close))
model = ARIMA(close, order=(5, 1, 2))
model_fit = model.fit()

print(model_fit.summary())

in_sample_pred = model_fit.predict(start=0, end=len(close)-1)

plt.figure(figsize=(12, 6))
plt.plot(dates, close, label='Actual')
plt.plot(dates, in_sample_pred, label='Predicted', alpha=0.7)
plt.title('ARIMA In-Sample Prediction') 
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()

In [ ]:
# GARCH
df['returns'] = df['close'].pct_change().dropna()*100
model = arch_model(df['returns'].dropna(), vol='Garch', p=1, q=1)
model_fit = model.fit()

print(model_fit.summary())

in_sample_pred = model_fit.conditional_volatility

oos_pred = model_fit.forecast(horizon=10)
forecast_variance = oos_pred.variance.iloc[-1]
forecast_volatility_oos = np.sqrt(forecast_variance)

df["historical_volatility"] = df['returns'].rolling(window=20).std()

future_dates = pd.date_range(start=df['datetime'].iloc[-1] + pd.Timedelta(minutes=5), periods=10, freq='5min')
plt.figure(figsize=(12, 6))
plt.plot(df['datetime'], df['historical_volatility'], label='Historical Volatility')
plt.plot(future_dates, forecast_volatility_oos.values, label='Forecasted Volatility', marker='o')

plt.plot(df['datetime'], in_sample_pred, label='In-Sample Volatility', alpha=0.7)
plt.title('GARCH Volatility Forecast')
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.legend()
plt.show()

In [ ]:
# SARIMA
# df.reset_index()
dates = df["date"]
close = df["close"]
close.index = range(len(close))
model = SARIMAX(close, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
model_fit = model.fit()

print(model_fit.summary())

in_sample_pred = model_fit.predict(start=0, end=len(close)-1)

plt.figure(figsize=(12, 6))
plt.plot(dates, close, label='Actual')
plt.plot(dates, in_sample_pred, label='Predicted', alpha=0.7)
plt.title('SARIMA In-Sample Prediction')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()

## Machine learning

In [ ]:
# Decision Tree Classifier
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
X = df.drop(columns=["close", "target"])
Y = df["target"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model = DecisionTreeClassifier()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)

accuracy = accuracy_score(Y_test, Y_pred)
print(f"Accuracy: {accuracy}")
print(f"Classification Report:\n{classification_report(Y_test, Y_pred)}")

In [ ]:
# Random forests
# df["SMA_10"]
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
X = df.drop(columns=["close", "target"])
Y = df["target"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model = RandomForestClassifier()
grid_search = GridSearchCV(estimator=model, param_grid={
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}, cv=3, n_jobs=-1, verbose=2)

grid_search.fit(X_train, Y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")

best_rf = grid_search.best_estimator_
Y_pred = best_rf.predict(X_test)
accuracy = accuracy_score(Y_test, Y_pred)
print(f"Accuracy: {accuracy}")
print(f"Classification Report:\n{classification_report(Y_test, Y_pred)}")

In [ ]:
# LSTM
scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(df.drop(columns=["close"]))
X_train = []
Y_train = []
window_size = 60
for i in range(window_size, len(X_scaled)):
    X_train.append(X_scaled[i-window_size:i, 0])
    Y_train.append(X_scaled[i, 0])
    
X_train, Y_train = np.array(X_train), np.array(Y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

model = Sequential()

model.add(LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(Dropout(0.2))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=1))

model.compile(optimizer='adam', loss='mean_squared_error')

model.fit(X_train, Y_train, epochs=50, batch_size=32)

df_test = df[-window_size:].copy()
df_total = pd.concat([df, df_test], axis=0)
inputs = df_total[len(df_total) - len(df_test) - window_size:]['close'].values
inputs = inputs.reshape(-1, 1)
inputs = scaler.transform(inputs)

X_test = []
for i in range(window_size, len(inputs)):
    X_test.append(inputs[i-window_size:i, 0])
    
X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

predicted_price = model.predict(X_test)
predicted_price = scaler.inverse_transform(predicted_price)

plt.figure(figsize=(12, 6))
plt.plot(df['datetime'], df['close'], label='Actual Price')
plt.plot(df_test['datetime'], predicted_price, label='Predicted Price', alpha=0.7)
plt.title('LSTM Price Prediction')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

rmse = np.sqrt(mean_squared_error(df_test['close'], predicted_price))
mae = mean_absolute_error(df_test['close'], predicted_price)
mape = mean_absolute_percentage_error(df_test['close'], predicted_price)

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"MAPE: {mape}")

In [ ]:
# Clustering
# Kmeans
X = df.drop(columns=["close"])
kmeans = KMeans(n_clusters=4, random_state=42)
df['cluster'] = kmeans.fit_predict(X)

sns.scatterplot(data=df, x='rsi', y='adx', hue='cluster', palette='Set1')
plt.title('Clustering de Indicadores Técnicos')
plt.xlabel('RSI')
plt.ylabel('ADX')
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df['datetime'], df['close'], label='Close Price')
for cluster in df['cluster'].unique():
    cluster_data = df[df['cluster'] == cluster]
    plt.scatter(cluster_data['datetime'], cluster_data['close'], label=f'Cluster {cluster}', alpha=0.6)
plt.title('Price Movement by Cluster')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()

In [ ]:
# DBSCAN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop(columns=["close"]))

dbscan = DBSCAN(eps=0.5, min_samples=5)
df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

plt.figure(figsize=(12, 6))
noise = df[df['dbscan_cluster'] == -1]
clusters = df[df['dbscan_cluster'] != -1]

plt.scatter(noise['datetime'], noise['close'], label='Noise', color='red', alpha=0.6)
plt.scatter(clusters['datetime'], clusters['close'], c=clusters['dbscan_cluster'], cmap='Set1', label='Clusters', alpha=0.6)
plt.title('DBSCAN Clustering of Price Movements')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()

# Backtesting

# Evaluación de resultados

In [ ]:
# Rendimiento total y anualizado

In [ ]:
# Ratio Sharpe


In [49]:
# Máximo drawdown


In [50]:
# Ratio Profit and Loss


In [51]:
# Costes operativos
